In [ ]:
%matplotlib inline

import tqdm
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import networkx as nx
import torch_geometric as pyg
import matplotlib.pyplot as plt
import torch.nn.functional as F

from tqdm import tqdm
from sklearn import metrics
from sklearn.manifold import TSNE
from torch_sparse import SparseTensor
from torch_geometric.utils import to_networkx
from torch.utils.data import DataLoader, SubsetRandomSampler

from common.utils import Utils
from common.nn_params import NNParams
from common.feed_fwd_nn import FeedFwdNN
from common.graph_conv_nn import GraphConvNN
from common.graph_sage_nn import GraphSageNN
from common.nn_model import NNModel, NNTrainParams
from common.graph_att_nn import GraphAttNN, GraphAttNNParams

In [ ]:
N_SAMPLES = 50_000
N_NEIGHBORS = [20, 15, 10]

DEFAULT_VAL_FIG_SIZE = (25, 20)
DEFAULT_VAL_TITLE_SIZE = 15
DEFAULT_VAL_LABEL_SIZE = 12

In [ ]:
dataset = pyg.datasets.Reddit2(
    root="./data"
    , transform=pyg.transforms.Compose(
        [
            pyg.transforms.NormalizeFeatures()
            , pyg.transforms.ToSparseTensor()
        ]
    )
)

In [ ]:
train_loader = pyg.loader.NeighborLoader(
    data=dataset._data
    , shuffle=True
    , num_workers=4
    , num_neighbors=N_NEIGHBORS
    , input_nodes=dataset._data.train_mask
    , batch_size=int(dataset._data.train_mask.sum())
)

val_loader = pyg.loader.NeighborLoader(
    data=dataset._data
    , shuffle=False
    , num_workers=4
    , num_neighbors=N_NEIGHBORS
    , input_nodes=dataset._data.val_mask
    , batch_size=int(dataset._data.val_mask.sum())
)

test_loader = pyg.loader.NeighborLoader(
    data=dataset._data
    , shuffle=False
    , num_workers=4
    , num_neighbors=N_NEIGHBORS
    , input_nodes=dataset._data.test_mask
    , batch_size=int(dataset._data.test_mask.sum())
)

print(f"Val Loader Len: {len(val_loader.dataset):,}")
print(f"Test Loader Len: {len(test_loader.dataset):,}")
print(f"Train Loader Len: {len(train_loader.dataset):,}")

In [ ]:
net = GraphSageNN(
    params=NNParams(
        dropout_prob=0.5
        , hidden_dims=[1024, 512, 256, 128]
        , input_dim=dataset.num_features
        , output_dim=dataset.num_classes
    )
)

model = NNModel(net=net, device="cpu")

In [ ]:
val_batch = next(iter(val_loader))
val_X, val_Y = net.unpack_batch(val_batch)
val_X, val_Y = tuple(x.numpy() for x in val_X), val_Y.numpy()

test_batch = next(iter(test_loader))
test_X, test_Y = net.unpack_batch(test_batch)
test_X, test_Y = tuple(x.numpy() for x in test_X), test_Y.numpy()

test_Y_hat_pre = model.predict(X=test_X)

In [ ]:
model_str, train_idps = model.train(
    params=NNTrainParams(
        optim="adam"
        , n_epochs=2000
        , snapshot_x=val_X
        , weight_decay=5e-4
        , learning_rate=1e-4
        , snapshot_interval=200
        , val_loader=val_loader
        , train_loader=train_loader
    )
)

trains = { model_str: (model, train_idps) }
top_model_names = [model_str]

In [ ]:
Utils.multi_line_plot(
    x_ticks_inc=50
    , y_axis_label="Loss"
    , x_axis_label="Iterations"
    , fig_size=DEFAULT_VAL_FIG_SIZE
    , label_size=DEFAULT_VAL_LABEL_SIZE
    , title_size=DEFAULT_VAL_TITLE_SIZE
    , title=f"Training & Validation Losses"
    , x=[idp.iter_idx for idp in trains[top_model_names[0]][1]]
    , yss_legend=[[f"{loss_type} of {model_name}" for loss_type in ["training", "validation"]] for model_name in top_model_names]
    , yss=[[[model_idp.train_loss for model_idp in trains[model_name][1]], [model_idp.val_loss for model_idp in trains[model_name][1]]] for model_name in top_model_names]
)

In [ ]:
Utils.multi_line_plot(
    x_ticks_inc=50
    , y_axis_label="Error"
    , x_axis_label="Iterations"
    , fig_size=DEFAULT_VAL_FIG_SIZE
    , label_size=DEFAULT_VAL_LABEL_SIZE
    , title_size=DEFAULT_VAL_TITLE_SIZE
    , title=f"Training & Validation Errors"
    , x=[idp.iter_idx for idp in trains[top_model_names[0]][1]]
    , yss_legend=[[f"{loss_type} of {model_name}" for loss_type in ["training", "validation"]] for model_name in top_model_names]
    , yss=[[[model_idp.train_error for model_idp in trains[model_name][1]], [model_idp.val_error for model_idp in trains[model_name][1]]] for model_name in top_model_names]
)

In [ ]:
print(f"{model_str} achieves validation error of {min(trains[top_model_names[0]][1], key=lambda idp: idp.val_error).val_error:.4f}")

In [ ]:
ts = [t for t in range(dataset.num_classes)]
cs = sns.color_palette(n_colors=dataset.num_classes)

In [ ]:
df_val_Y = pd.DataFrame(data=val_Y, columns=["community"])

for train_idp in train_idps:
    if train_idp.snapshot_y_hat is None:
        continue

    print(f"Val data accuracy score @ epoch_idx={train_idp.epoch_idx}: ", metrics.accuracy_score(y_true=val_Y, y_pred=train_idp.snapshot_y_hat[1]))
    print(f"Val data f1 score @ epoch_idx={train_idp.epoch_idx}: ", metrics.f1_score(y_true=val_Y, y_pred=train_idp.snapshot_y_hat[1], average="micro", zero_division=0))
    print(f"Val data recall score @ epoch_idx={train_idp.epoch_idx}: ", metrics.recall_score(y_true=val_Y, y_pred=train_idp.snapshot_y_hat[1], average="micro", zero_division=0))
    print(f"Val data precision score @ epoch_idx={train_idp.epoch_idx}: ", metrics.precision_score(y_true=val_Y, y_pred=train_idp.snapshot_y_hat[1], average="micro", zero_division=0))
    print("-" * 150)
    
    Utils.scatter_plot(
        figsize=DEFAULT_VAL_FIG_SIZE
        , title_size=DEFAULT_VAL_TITLE_SIZE
        , label_size=DEFAULT_VAL_LABEL_SIZE
        , vm=Utils.get_scatter_plot_vm(
            data=pd.concat(
                axis=1
                , objs=[
                    pd.DataFrame(
                        data=TSNE(
                            n_components=2
                        ).fit_transform(
                            X=pd.DataFrame(
                                data=train_idp.snapshot_y_hat[0]
                            ).iloc[:N_SAMPLES, :]
                        )
                        , columns=["tsne_1", "tsne_2"]
                    )
                    , df_val_Y.iloc[:N_SAMPLES, :]
                ]
            )
            , uni_ts=ts
            , colors_ts=cs
            , labels_ts=ts
            , col_xs="tsne_1"
            , col_ys="tsne_2"
            , label_xs="tsne_1"
            , label_ys="tsne_2"
            , col_ts="community"
            , title=f"t-SNE projected 2D visualization of predicted output logits of validation data snapshot using {model_str} @ epoch_idx={train_idp.epoch_idx}"
        )
    )

val_Y_hat_post = model.predict(X=val_X)

print("Val data accuracy score: ", metrics.accuracy_score(y_true=val_Y, y_pred=val_Y_hat_post[1]))
print("Val data f1 score: ", metrics.f1_score(y_true=val_Y, y_pred=val_Y_hat_post[1], average="micro", zero_division=0))
print("Val data recall score: ", metrics.recall_score(y_true=val_Y, y_pred=val_Y_hat_post[1], average="micro", zero_division=0))
print("Val data precision score: ", metrics.precision_score(y_true=val_Y, y_pred=val_Y_hat_post[1], average="micro", zero_division=0))

In [ ]:
df_test_Y = pd.DataFrame(data=test_Y, columns=["community"])

test_Y_hat_post = model.predict(X=test_X)

print("Test data accuracy score post training: ", metrics.accuracy_score(y_true=test_Y, y_pred=test_Y_hat_post[1]))
print("Test data f1 score post training: ", metrics.f1_score(y_true=test_Y, y_pred=test_Y_hat_post[1], average="micro", zero_division=0))
print("Test data recall score post training: ", metrics.recall_score(y_true=test_Y, y_pred=test_Y_hat_post[1], average="micro", zero_division=0))
print("Test data precision score post training: ", metrics.precision_score(y_true=test_Y, y_pred=test_Y_hat_post[1], average="micro", zero_division=0))

for at, test_Y_hat in [("pre", test_Y_hat_pre), ("post", test_Y_hat_post)]:
    Utils.scatter_plot(
        figsize=DEFAULT_VAL_FIG_SIZE
        , title_size=DEFAULT_VAL_TITLE_SIZE
        , label_size=DEFAULT_VAL_LABEL_SIZE
        , vm=Utils.get_scatter_plot_vm(
            data=pd.concat(
                axis=1
                , objs=[
                    pd.DataFrame(
                        data=TSNE(
                            n_components=2
                        ).fit_transform(
                            X=pd.DataFrame(
                                data=test_Y_hat[0]
                            ).iloc[:N_SAMPLES, :]
                        )
                        , columns=["tsne_1", "tsne_2"]
                    )
                    , df_test_Y.iloc[:N_SAMPLES, :]
                ]
            )
            , uni_ts=ts
            , labels_ts=ts
            , colors_ts=cs
            , col_xs="tsne_1"
            , col_ys="tsne_2"
            , label_xs="tsne_1"
            , label_ys="tsne_2"
            , col_ts="community"
            , title=f"t-SNE projected 2D visualization of predicted output logits of test data using {model_str} {at} training"
        )
    )